# GACS Dataset Builder

**Purpose**: Build the GACS affective dataset from raw videos listed in `video_manifest.csv`.

## Pipeline Overview
1. Load video manifest
2. Perform scene detection (PySceneDetect)
3. Extract one keyframe per scene (mid-frame)
4. Label keyframes with Claude Vision API (mood, style, objects)
5. Generate embeddings using SBERT
6. Save final dataset to `data/embeddings/gacs_dataset.csv`

## Output Schema
- `data/scenes/{video_id}/{scene_id}.jpg` - Extracted keyframes
- `data/annotations/{image_id}.json` - Raw annotations
- `data/embeddings/gacs_dataset.csv` - Final embedded dataset

## 1. Setup & Configuration

In [ ]:
# Install dependencies (run once)
# !pip install opencv-python scenedetect anthropic numpy pandas Pillow matplotlib tqdm sentence-transformers

In [ ]:
import base64
import hashlib
import json
import time
from dataclasses import asdict, dataclass, field
from datetime import datetime
from pathlib import Path
from typing import List, Optional, Tuple

import anthropic
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scenedetect import ContentDetector, SceneManager, open_video
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm

from gacs_config import (
    ANNOTATIONS_DIR,
    API_RATE_LIMIT_DELAY,
    CLAUDE_MODEL,
    DATA_DIR,
    EMBEDDINGS_DIR,
    MIN_SCENE_LENGTH,
    PROJECT_ROOT,
    PROMPT_VERSION,
    QUICK_TEST_MAX_VIDEOS,
    QUICK_TEST_MODE,
    SBERT_MODEL,
    SCENE_THRESHOLD,
    SCENES_DIR,
    setup_logging,
    validate_config,
)

logger = setup_logging("gacs_dataset_builder")

print("All dependencies loaded successfully!")

In [ ]:
# Configuration (imported from gacs_config)
BASE_DIR = PROJECT_ROOT

# Validate configuration
validate_config()

print(f"Base directory: {BASE_DIR}")
print(f"Data directory: {DATA_DIR}")

In [ ]:
# Load video manifest
manifest_path = BASE_DIR / "video_manifest.csv"
manifest_df = pd.read_csv(manifest_path)

# Fix path separators for cross-platform compatibility using pathlib
manifest_df['local_path'] = manifest_df['local_path'].apply(lambda p: str(Path(p)))

# Rename columns if needed to match expected schema
if 'local_path' in manifest_df.columns and 'file_path' not in manifest_df.columns:
    manifest_df['file_path'] = manifest_df['local_path']

logger.info(f"Loaded {len(manifest_df)} videos from manifest")
print(f"Loaded {len(manifest_df)} videos from manifest")
print(f"Columns: {list(manifest_df.columns)}")
display(manifest_df.head())

## 2. Data Classes & Schemas

In [ ]:
@dataclass
class Scene:
    """Represents a detected scene in a video."""
    scene_id: str
    video_id: str
    scene_index: int
    start_frame: int
    end_frame: int
    start_time: float
    end_time: float
    duration: float
    keyframe_path: Optional[str] = None

@dataclass
class Annotation:
    """Annotation schema for a keyframe image."""
    image_id: str
    video_id: str
    scene_id: str
    mood_words: List[str]
    style_words: List[str]
    object_words: List[str]
    labeling_model: str
    prompt_version: str
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())

@dataclass
class GACSEntry:
    """Final GACS dataset entry with embeddings."""
    image_id: str
    video_id: str
    scene_id: str
    keyframe_path: str
    mood_vector: List[float]
    mood_words: str  # comma-separated
    style_words: str  # comma-separated
    object_words: str  # comma-separated

print("Data classes defined.")

## 3. Scene Detection Module

In [ ]:
def detect_scenes(video_path: Path, video_id: str,
                  threshold: float = SCENE_THRESHOLD,
                  min_scene_len: int = MIN_SCENE_LENGTH) -> List[Scene]:
    """
    Detect scene boundaries in a video using PySceneDetect's ContentDetector.
    
    Args:
        video_path: Path to the video file
        video_id: Unique identifier for the video
        threshold: Scene change detection threshold
        min_scene_len: Minimum number of frames per scene
    
    Returns:
        List of Scene objects
    """
    if not video_path.exists():
        logger.warning(f"Video not found at {video_path}")
        return []

    # Open video and create scene manager
    video = open_video(str(video_path))
    scene_manager = SceneManager()
    scene_manager.add_detector(ContentDetector(threshold=threshold, min_scene_len=min_scene_len))

    # Detect scenes
    scene_manager.detect_scenes(video)
    scene_list = scene_manager.get_scene_list()

    # Convert to Scene objects
    scenes = []
    for idx, (start, end) in enumerate(scene_list):
        scene_id = f"{video_id}_s{idx:04d}"
        scene = Scene(
            scene_id=scene_id,
            video_id=video_id,
            scene_index=idx,
            start_frame=start.get_frames(),
            end_frame=end.get_frames(),
            start_time=start.get_seconds(),
            end_time=end.get_seconds(),
            duration=end.get_seconds() - start.get_seconds()
        )
        scenes.append(scene)

    # Save scene metadata to JSON
    scene_dir = SCENES_DIR / video_id
    scene_dir.mkdir(parents=True, exist_ok=True)
    metadata_path = scene_dir / "scene_metadata.json"
    scene_dicts = [asdict(s) for s in scenes]
    with open(metadata_path, 'w') as f:
        json.dump(scene_dicts, f, indent=2)
    logger.info(f"Saved scene metadata to {metadata_path}")

    return scenes

print("Scene detection function defined.")

## 4. Keyframe Extraction Module

In [ ]:
def extract_keyframes(video_path: Path, scenes: List[Scene], video_id: str) -> List[Scene]:
    """
    Extract mid-frame keyframes from each scene.
    Saves to: data/scenes/{video_id}/{scene_id}.jpg
    
    Args:
        video_path: Path to the video file
        scenes: List of Scene objects
        video_id: Video identifier
    
    Returns:
        List of Scene objects with keyframe_path populated
    """
    # Create output directory
    output_dir = SCENES_DIR / video_id
    output_dir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        logger.error(f"Could not open video {video_path}")
        return scenes

    updated_scenes = []
    for scene in scenes:
        # Calculate mid-frame
        mid_frame = (scene.start_frame + scene.end_frame) // 2

        # Extract frame
        cap.set(cv2.CAP_PROP_POS_FRAMES, mid_frame)
        ret, frame = cap.read()

        if ret:
            # Save keyframe as {scene_id}.jpg
            keyframe_filename = f"{scene.scene_id}.jpg"
            keyframe_path = output_dir / keyframe_filename
            cv2.imwrite(str(keyframe_path), frame)

            # Store relative path
            scene.keyframe_path = str(keyframe_path.relative_to(BASE_DIR))

        updated_scenes.append(scene)

    cap.release()
    return updated_scenes

print("Keyframe extraction function defined.")

## 5. Claude Vision API Labeling

In [ ]:
# Initialize Claude client
client = anthropic.Anthropic()  # Uses ANTHROPIC_API_KEY environment variable

# Exact prompts from specification
MOOD_PROMPT = """You are an affective labeling engine.
List exactly five adjectives that best describe the emotional mood.
Rules:
- Only adjectives.
- No sentences.
- No explanations.
- No duplicates.
- Avoid generic words.
Output format:
adj1, adj2, adj3, adj4, adj5"""

STYLE_PROMPT = """You are a visual style classifier.
List exactly three short visual style descriptors.
Rules:
- Under 12 characters each.
- No emotions.
- No sentences.
Output format:
style1, style2, style3"""

OBJECT_PROMPT = """You are an object extraction system.
List 3 to 5 concrete object nouns visible.
Rules:
- Only nouns.
- No abstract concepts.
- No sentences.
Output format:
obj1, obj2, obj3, obj4, obj5"""

print("Claude API prompts defined.")

In [ ]:
def encode_image_base64(image_path: Path) -> str:
    """Encode image to base64 for Claude API."""
    with open(image_path, "rb") as f:
        return base64.standard_b64encode(f.read()).decode("utf-8")

def get_image_media_type(image_path: Path) -> str:
    """Get media type from image extension."""
    ext = image_path.suffix.lower()
    media_types = {
        '.jpg': 'image/jpeg',
        '.jpeg': 'image/jpeg',
        '.png': 'image/png',
        '.gif': 'image/gif',
        '.webp': 'image/webp'
    }
    return media_types.get(ext, 'image/jpeg')

def call_claude_vision(image_path: Path, prompt: str, max_retries: int = 3) -> Optional[str]:
    """
    Call Claude Vision API with an image and prompt.
    Includes exponential backoff for rate limit errors.
    
    Args:
        image_path: Path to the image
        prompt: The prompt to send
        max_retries: Maximum number of retry attempts
    
    Returns:
        Response text or None if failed
    """
    for attempt in range(max_retries):
        try:
            image_data = encode_image_base64(image_path)
            media_type = get_image_media_type(image_path)

            message = client.messages.create(
                model=CLAUDE_MODEL,
                max_tokens=256,
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "image",
                                "source": {
                                    "type": "base64",
                                    "media_type": media_type,
                                    "data": image_data
                                }
                            },
                            {
                                "type": "text",
                                "text": prompt
                            }
                        ]
                    }
                ]
            )

            return message.content[0].text.strip()

        except anthropic.RateLimitError:
            wait = 2 ** attempt * API_RATE_LIMIT_DELAY
            logger.warning(f"Rate limited, waiting {wait}s (attempt {attempt+1}/{max_retries})")
            time.sleep(wait)
        except Exception as e:
            logger.error(f"API error: {e}")
            if attempt < max_retries - 1:
                time.sleep(API_RATE_LIMIT_DELAY)
            else:
                return None
    return None

def parse_comma_list(text: str) -> List[str]:
    """Parse comma-separated response into list."""
    if not text:
        return []
    # Clean and split
    items = [item.strip().lower() for item in text.split(',')]
    # Remove empty items
    items = [item for item in items if item]
    return items

def validate_labels(mood_words: List[str], style_words: List[str], object_words: List[str]) -> List[str]:
    """Validate label counts match specification."""
    issues = []
    if len(mood_words) != 5:
        issues.append(f"Expected 5 mood words, got {len(mood_words)}")
    if len(style_words) != 3:
        issues.append(f"Expected 3 style words, got {len(style_words)}")
    if not (3 <= len(object_words) <= 5):
        issues.append(f"Expected 3-5 object words, got {len(object_words)}")
    return issues

print("Claude Vision API functions defined.")

In [ ]:
def label_keyframe(image_path: Path, video_id: str, scene_id: str, max_label_retries: int = 2) -> Optional[Annotation]:
    """
    Label a single keyframe with mood, style, and object words.
    Retries on validation failure up to max_label_retries times.
    
    Args:
        image_path: Path to the keyframe image
        video_id: Video identifier
        scene_id: Scene identifier
        max_label_retries: Maximum number of retries on label validation failure
    
    Returns:
        Annotation object or None if failed
    """
    # Generate image_id from path hash
    image_id = hashlib.md5(str(image_path).encode()).hexdigest()[:12]

    for attempt in range(max_label_retries + 1):
        # Call Claude for each prompt type
        mood_response = call_claude_vision(image_path, MOOD_PROMPT)
        time.sleep(API_RATE_LIMIT_DELAY)

        style_response = call_claude_vision(image_path, STYLE_PROMPT)
        time.sleep(API_RATE_LIMIT_DELAY)

        object_response = call_claude_vision(image_path, OBJECT_PROMPT)

        if not all([mood_response, style_response, object_response]):
            logger.warning(f"Incomplete labeling for {image_path}")
            return None

        # Parse responses
        mood_words = parse_comma_list(mood_response)
        style_words = parse_comma_list(style_response)
        object_words = parse_comma_list(object_response)

        issues = validate_labels(mood_words, style_words, object_words)
        if not issues:
            break
        logger.warning(f"Label validation failed (attempt {attempt+1}): {issues}")

    # Proceed even if imperfect after retries
    return Annotation(
        image_id=image_id,
        video_id=video_id,
        scene_id=scene_id,
        mood_words=mood_words,
        style_words=style_words,
        object_words=object_words,
        labeling_model=CLAUDE_MODEL,
        prompt_version=PROMPT_VERSION
    )

def save_annotation(annotation: Annotation):
    """Save annotation to data/annotations/{image_id}.json"""
    annotation_path = ANNOTATIONS_DIR / f"{annotation.image_id}.json"
    with open(annotation_path, 'w') as f:
        json.dump(asdict(annotation), f, indent=2)
    return annotation_path

def load_annotation(image_id: str) -> Optional[Annotation]:
    """Load annotation from cache if exists."""
    annotation_path = ANNOTATIONS_DIR / f"{image_id}.json"
    if annotation_path.exists():
        with open(annotation_path, 'r') as f:
            data = json.load(f)
        return Annotation(**data)
    return None

print("Labeling functions defined.")

## 6. Embedding Generation (SBERT)

In [ ]:
# Initialize SBERT model
print("Loading SBERT model...")
sbert_model = SentenceTransformer(SBERT_MODEL)
print(f"SBERT model '{SBERT_MODEL}' loaded.")

In [ ]:
def generate_mood_embedding(mood_words: List[str]) -> List[float]:
    """
    Generate mood vector by averaging SBERT embeddings of mood words.
    
    Args:
        mood_words: List of mood adjectives
    
    Returns:
        Averaged embedding vector as list of floats
    """
    if not mood_words:
        return [0.0] * 384  # SBERT dimension

    # Get embeddings for each word
    embeddings = sbert_model.encode(mood_words)

    # Average the embeddings
    mood_vector = np.mean(embeddings, axis=0)

    return mood_vector.tolist()

print("Embedding generation function defined.")

## 7. Full Pipeline Execution

In [ ]:
def process_video(video_row: pd.Series, skip_existing: bool = True) -> List[Tuple[Scene, Annotation]]:
    """
    Process a single video through the full pipeline.
    
    Args:
        video_row: Row from the video manifest DataFrame
        skip_existing: Skip scenes that already have annotations
    
    Returns:
        List of (scene, annotation) tuples
    """
    video_id = video_row['video_id']
    video_path = BASE_DIR / video_row['file_path']

    logger.info(f"Processing video: {video_id}")
    print(f"\n{'='*60}")
    print(f"Processing: {video_id}")
    print(f"{'='*60}")

    # Check if video exists
    if not video_path.exists():
        logger.warning(f"Video not found at {video_path}")
        return []

    # Step 1: Detect scenes
    print("\n[1/4] Detecting scenes...")
    scenes = detect_scenes(video_path, video_id)
    logger.info(f"Found {len(scenes)} scenes in {video_id}")
    print(f"  Found {len(scenes)} scenes")

    if not scenes:
        print("  No scenes detected, skipping video.")
        return []

    # Step 2: Extract keyframes
    print("\n[2/4] Extracting keyframes...")
    scenes = extract_keyframes(video_path, scenes, video_id)
    keyframe_count = sum(1 for s in scenes if s.keyframe_path)
    logger.info(f"Extracted {keyframe_count} keyframes for {video_id}")
    print(f"  Extracted {keyframe_count} keyframes")

    # Step 3: Label with Claude API
    print("\n[3/4] Labeling with Claude API...")
    results = []

    for scene in tqdm(scenes, desc="Labeling"):
        if not scene.keyframe_path:
            continue

        keyframe_path = BASE_DIR / scene.keyframe_path
        image_id = hashlib.md5(str(keyframe_path).encode()).hexdigest()[:12]

        # Check cache (resume capability)
        if skip_existing:
            cached = load_annotation(image_id)
            if cached:
                results.append((scene, cached))
                continue

        # Label keyframe
        annotation = label_keyframe(keyframe_path, video_id, scene.scene_id)

        if annotation:
            save_annotation(annotation)
            results.append((scene, annotation))

        time.sleep(API_RATE_LIMIT_DELAY)

    logger.info(f"Labeled {len(results)} scenes for {video_id}")
    print(f"  Labeled {len(results)} scenes")

    return results

print("Pipeline function defined.")

In [ ]:
# Process all videos in manifest
all_results = []

# Filter to only downloaded videos
available_videos = manifest_df[manifest_df['download_ok'] == True]

if QUICK_TEST_MODE:
    available_videos = available_videos.head(QUICK_TEST_MAX_VIDEOS)
    logger.info(f"QUICK_TEST_MODE: Processing only {len(available_videos)} video(s)")

print(f"Processing {len(available_videos)} available videos...\n")

for idx, row in available_videos.iterrows():
    results = process_video(row, skip_existing=True)
    all_results.extend(results)

print(f"\n{'='*60}")
print("Scene Processing Complete!")
print(f"{'='*60}")
print(f"Total scenes processed: {len(all_results)}")
logger.info(f"Pipeline complete: {len(all_results)} scenes processed")

## 8. Generate Final Dataset

In [ ]:
def build_gacs_dataset(results: List[Tuple[Scene, Annotation]]) -> pd.DataFrame:
    """
    Build final GACS dataset with embeddings.
    
    Args:
        results: List of (scene, annotation) tuples
    
    Returns:
        DataFrame with GACS dataset
    """
    print("\n[4/4] Generating embeddings and building dataset...")

    dataset = []

    for scene, annotation in tqdm(results, desc="Building dataset"):
        # Generate mood embedding
        mood_vector = generate_mood_embedding(annotation.mood_words)

        entry = GACSEntry(
            image_id=annotation.image_id,
            video_id=annotation.video_id,
            scene_id=annotation.scene_id,
            keyframe_path=scene.keyframe_path or "",
            mood_vector=mood_vector,
            mood_words=', '.join(annotation.mood_words),
            style_words=', '.join(annotation.style_words),
            object_words=', '.join(annotation.object_words)
        )
        dataset.append(asdict(entry))

    df = pd.DataFrame(dataset)

    # Convert mood_vector list to string for CSV storage
    df['mood_vector'] = df['mood_vector'].apply(lambda x: ','.join(map(str, x)))

    return df

# Build and save dataset
if all_results:
    gacs_df = build_gacs_dataset(all_results)

    # Save to CSV
    output_path = EMBEDDINGS_DIR / "gacs_dataset.csv"
    gacs_df.to_csv(output_path, index=False)

    print(f"\nDataset saved to: {output_path}")
    print(f"Total entries: {len(gacs_df)}")
    print(f"\nColumns: {list(gacs_df.columns)}")
    display(gacs_df.head())
else:
    print("No results to save. Make sure videos are available and processed.")

## 9. Dataset Validation & Statistics

In [ ]:
def validate_dataset(df: pd.DataFrame):
    """
    Validate and display statistics for the GACS dataset.
    """
    print("\n" + "="*60)
    print("Dataset Validation")
    print("="*60)

    # Basic stats
    print(f"\nTotal entries: {len(df)}")
    print(f"Unique videos: {df['video_id'].nunique()}")
    print(f"Unique scenes: {df['scene_id'].nunique()}")

    # Check for missing values
    print("\nMissing values:")
    for col in df.columns:
        missing = df[col].isna().sum()
        if missing > 0:
            print(f"  {col}: {missing}")

    # Mood word frequency
    print("\nTop 20 mood words:")
    all_mood_words = []
    for words in df['mood_words']:
        all_mood_words.extend([w.strip() for w in words.split(',')])

    mood_counts = pd.Series(all_mood_words).value_counts().head(20)
    print(mood_counts)

    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Mood word frequency
    mood_counts.plot(kind='barh', ax=axes[0], color='steelblue')
    axes[0].set_xlabel('Count')
    axes[0].set_title('Top 20 Mood Words')
    axes[0].invert_yaxis()

    # Scenes per video
    scenes_per_video = df.groupby('video_id').size()
    scenes_per_video.plot(kind='bar', ax=axes[1], color='coral')
    axes[1].set_xlabel('Video ID')
    axes[1].set_ylabel('Number of Scenes')
    axes[1].set_title('Scenes per Video')
    axes[1].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.savefig(EMBEDDINGS_DIR / 'dataset_statistics.png', dpi=150, bbox_inches='tight')
    plt.show()

# Validate if dataset exists
if 'gacs_df' in dir() and len(gacs_df) > 0:
    validate_dataset(gacs_df)

In [ ]:
def visualize_mood_embeddings(df: pd.DataFrame):
    """
    Visualize mood embeddings using dimensionality reduction.
    """
    from sklearn.manifold import TSNE

    # Parse mood vectors back to arrays
    vectors = np.array([list(map(float, v.split(','))) for v in df['mood_vector']])

    # Apply t-SNE
    print("Applying t-SNE dimensionality reduction...")
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(vectors)-1))
    vectors_2d = tsne.fit_transform(vectors)

    # Plot
    fig, ax = plt.subplots(figsize=(12, 8))

    # Color by video
    video_ids = df['video_id'].unique()
    colors = plt.cm.tab10(np.linspace(0, 1, len(video_ids)))
    color_map = dict(zip(video_ids, colors))

    for vid in video_ids:
        mask = df['video_id'] == vid
        ax.scatter(vectors_2d[mask, 0], vectors_2d[mask, 1],
                   c=[color_map[vid]], label=vid[:25], alpha=0.7, s=60)

    ax.set_xlabel('t-SNE Dimension 1')
    ax.set_ylabel('t-SNE Dimension 2')
    ax.set_title('GACS Mood Embeddings (t-SNE)')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)

    plt.tight_layout()
    plt.savefig(EMBEDDINGS_DIR / 'mood_embeddings_tsne.png', dpi=150, bbox_inches='tight')
    plt.show()

# Visualize if dataset exists
if 'gacs_df' in dir() and len(gacs_df) > 5:
    visualize_mood_embeddings(gacs_df)

## 10. Summary

This notebook has:
1. Loaded video manifest from `video_manifest.csv`
2. Detected scene boundaries using PySceneDetect
3. Extracted mid-frame keyframes to `data/scenes/{video_id}/{scene_id}.jpg`
4. Labeled keyframes with Claude Vision API:
   - Mood words (5 adjectives)
   - Style words (3 descriptors)
   - Object words (3-5 nouns)
5. Saved annotations to `data/annotations/{image_id}.json`
6. Generated SBERT embeddings for mood words
7. Saved final dataset to `data/embeddings/gacs_dataset.csv`

**Output Schema (gacs_dataset.csv):**
- `image_id` - Unique identifier for the keyframe
- `video_id` - Source video identifier
- `scene_id` - Scene identifier
- `keyframe_path` - Relative path to keyframe image
- `mood_vector` - SBERT embedding (384-dim, comma-separated)
- `mood_words` - Comma-separated mood adjectives
- `style_words` - Comma-separated style descriptors
- `object_words` - Comma-separated object nouns

**Features:**
- Progress bars for all operations
- Resume capability (skips existing annotations)
- Error handling for API failures
- Colab-compatible